# KPI lab: how good is Thunder-X?

For each benchmark row: send the instruction to Thunder-X, get its XPath back, and check it on the page.
The answer is right when its XPath points to exactly one element, and it's the same element as `expected_xpath`.

Start Thunder-X first: `./run.sh`

In [2]:
import asyncio
import sqlite3
import time

import httpx
from browser_use import BrowserSession

THUNDERX = "http://localhost:8000"
DB = "../data/bench/bench_20261101_231635.sqlite"

## Load the benchmark rows, grouped by page

In [3]:
connection = sqlite3.connect(DB)
connection.row_factory = sqlite3.Row
rows = []
for row in connection.execute("SELECT * FROM samples"):
    rows.append(dict(row))
connection.close()

pages = {}
for row in rows:
    url = row["url"]
    if url not in pages:
        pages[url] = []
    pages[url].append(row)

print(len(rows), "rows on", len(pages), "pages")

178 rows on 11 pages


## Helpers

- `ask_thunderx`: send one instruction, get Thunder-X's answer and how long it took
- `is_right`: check Thunder-X's XPath against the expected one, on the page

In [4]:
#NB :position is the step's place in the session's chain: 1, 2, 3...

In [5]:
http = httpx.Client(base_url=THUNDERX, timeout=120)


def ask_thunderx(session_id, position, instruction):
    start = time.time()
    request = {"session_id": session_id, "position": position, "instruction": instruction, "mode": "validate"}
    answer = http.post("/run", json=request).json()
    seconds = time.time() - start
    return answer, seconds
    
#const found    = document.evaluate(xpath, document, null, 7, null);           // run Thunder-X's XPath
#const expected = document.evaluate(expected_xpath, document, null, 7, null);  // run the expected XPath
# document.evaluate is the browser's own XPath engine. The 7 asks for the list of every element the XPath matches.
# more than 1 ambigious 
# match

SAME_ELEMENT_JS = """(xpath, expected_xpath) => {
  const found = document.evaluate(xpath, document, null, 7, null);
  const expected = document.evaluate(expected_xpath, document, null, 7, null);
  if (found.snapshotLength !== 1) return false;       
  return found.snapshotItem(0) === expected.snapshotItem(0);
}"""

#snapshotItem(0) is the element that was found. === checks whether it's the very


async def is_right(page, xpath, expected_xpath):
    if expected_xpath == "NONE":
        return xpath is None
    if xpath is None:
        return False
    try:
        answer = await page.evaluate(SAME_ELEMENT_JS, xpath, expected_xpath)
    except Exception:
        return False
    return answer == "True"

## Run

For each page: Thunder-X opens it, and the lab opens it too (to check the answers).
Then each row of that page is sent to Thunder-X, one by one.

In [9]:
browser = BrowserSession(headless=True, keep_alive=True)
await browser.start()
results = []

for url in pages:
    print("page:", url)
    session_id = http.post("/sessions", json={"url": url, "title": "kpi lab"}).json()["session_id"]
    await browser.navigate_to(url)
    await asyncio.sleep(2)
    page = await browser.get_current_page()

    position = 1
    for row in pages[url]:
        answer, seconds = ask_thunderx(session_id, position, row["instruction"])
        right = await is_right(page, answer.get("xpath"), row["expected_xpath"])

        result = {
            "instruction": row["instruction"],
            "category": row["category"],
            "expected_action": row["action"],
            "expected_value": row["value"],
            "expected_xpath": row["expected_xpath"],
            "action": answer.get("action"),
            "value": answer.get("value"),
            "xpath": answer.get("xpath"),
            "status": answer.get("status"),
            "route": answer.get("route"),
            "confidence": answer.get("confidence"),
            "seconds": seconds,
            "right": right,
        }
        results.append(result)

        if right:
            print("   right", round(seconds, 1), "s", row["instruction"])
        else:
            print("   WRONG", round(seconds, 1), "s", row["instruction"])
        position = position + 1

    http.delete("/sessions/" + session_id)

await browser.kill()

INFO     [BrowserSession] Setting viewport to 1920x1200 with device scale factor 1.0 whereas original device scale factor was 1.0


page: https://www.selenium.dev/selenium/web/web-form.html
   WRONG 2.8 s Find the text 'Web form'
   WRONG 3.5 s Locate 'Web form'
   right 0.9 s Fill 'Text input' with 'Jane Doe'
   right 1.0 s Enter 'Jane Doe' in 'Text input'
   right 0.1 s Click on 'Submit'
   right 0.2 s Press 'Submit'
   right 2.0 s Double-click on 'File input'
   right 3.1 s Double click the 'File input' button
   right 0.1 s Right-click on 'Return to index'
   right 0.1 s Right click the 'Return to index' link
   right 2.5 s Choose 'One' from 'Dropdown (select)'
   right 2.4 s Select 'One' in the 'Dropdown (select)' dropdown
   WRONG 2.0 s Check 'Checked radio'
   WRONG 2.4 s Tick the 'Checked radio' box
   right 1.8 s Enter 'hello world' in 'Readonly input'
   right 1.9 s Type 'hello world' into the 'Readonly input' field
   WRONG 1.9 s Select the 'Default radio' option
   WRONG 1.8 s Tick the 'Default radio' box
page: https://testpages.eviltester.com/styled/basic-html-form-test.html
   WRONG 1.1 s Click the 'T

INFO     [BrowserSession] 📢 on_BrowserStopEvent - Calling reset() (force=True, keep_alive=True)
INFO     [BrowserSession] [SessionManager] Cleared all owned data (targets, sessions, mappings)
INFO     [BrowserSession] ✅ Browser session reset complete
INFO     [BrowserSession] ✅ Browser session reset complete


## KPIs

- **accuracy**: right answers / all rows
- **coverage**: rows where Thunder-X gave an XPath / all rows
- **precision**: right answers / rows where Thunder-X gave an XPath
- **action accuracy**: Thunder-X read the right action from the instruction
- **value accuracy**: Thunder-X read the right value (only rows that have one)
- **time**: seconds per instruction

In [13]:
def percent(part, whole):
    if whole == 0:
        return "-"
    return str(round(100 * part / whole)) + " %"


total = len(results)
right = 0
answered = 0
right_when_answered = 0
right_action = 0
with_value = 0
right_value = 0
times = []

for r in results:
    times.append(r["seconds"])
    if r["right"]:
        right = right + 1
    if r["xpath"] is not None:
        answered = answered + 1
        if r["right"]:
            right_when_answered = right_when_answered + 1
    if r["action"] == r["expected_action"]:
        right_action = right_action + 1
    if r["expected_value"] is not None:
        with_value = with_value + 1
        if str(r["value"]).lower() == str(r["expected_value"]).lower():
            right_value = right_value + 1

times.sort()
print("rows           ", total)
print("accuracy       ", percent(right, total))
print("coverage       ", percent(answered, total))
print("precision      ", percent(right_when_answered, answered))
print("action accuracy", percent(right_action, total))
print("time average   ", round(sum(times) / total, 2), "s")
print("time median    ", round(times[total // 2], 2), "s")
print("time slowest   ", round(times[-1], 2), "s")

rows            178
accuracy        67 %
coverage        98 %
precision       68 %
action accuracy 78 %
time average    0.78 s
time median     0.54 s
time slowest    3.51 s


## Wrong answers

In [14]:
for r in results:
    if not r["right"]:
        print(r["instruction"])
        print("   expected:", r["expected_xpath"])
        print("   got:     ", r["xpath"], "|", r["status"], "|", r["route"])

Find the text 'Web form'
   expected: //h1[normalize-space()='Web form']
   got:      None | not_found | laya
Locate 'Web form'
   expected: //h1[normalize-space()='Web form']
   got:      //a[normalize-space()='Return to index'] | verified | laya
Check 'Checked radio'
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[2]/label[1]/input[1]
   got:      //label[normalize-space()='Checked radio'] | verified | laya
Tick the 'Checked radio' box
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[2]/label[1]/input[1]
   got:      //label[normalize-space()='Checked radio'] | verified | laya
Select the 'Default radio' option
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[3]/label[1]/input[1]
   got:      //label[normalize-space()='Default radio'] | verified | laya
Tick the 'Default radio' box
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[3]/label[1]/input[1]
   got:      //label[normalize-space()='Default 

# RUN 2 , thunder x , but this time laya browser 

In [5]:
browser = BrowserSession(headless=True, keep_alive=True)
await browser.start()
results = []

for url in pages:
    print("page:", url)
    session_id = http.post("/sessions", json={"url": url, "title": "kpi lab"}).json()["session_id"]
    await browser.navigate_to(url)
    await asyncio.sleep(2)
    page = await browser.get_current_page()

    position = 1
    for row in pages[url]:
        answer, seconds = ask_thunderx(session_id, position, row["instruction"])
        right = await is_right(page, answer.get("xpath"), row["expected_xpath"])

        result = {
            "instruction": row["instruction"],
            "category": row["category"],
            "expected_action": row["action"],
            "expected_value": row["value"],
            "expected_xpath": row["expected_xpath"],
            "action": answer.get("action"),
            "value": answer.get("value"),
            "xpath": answer.get("xpath"),
            "status": answer.get("status"),
            "route": answer.get("route"),
            "confidence": answer.get("confidence"),
            "seconds": seconds,
            "right": right,
        }
        results.append(result)

        if right:
            print("   right", round(seconds, 1), "s", row["instruction"])
        else:
            print("   WRONG", round(seconds, 1), "s", row["instruction"])
        position = position + 1

    http.delete("/sessions/" + session_id)

await browser.kill()

INFO     [BrowserSession] Setting viewport to 1920x1200 with device scale factor 1.0 whereas original device scale factor was 1.0


page: https://www.selenium.dev/selenium/web/web-form.html
   WRONG 2.4 s Find the text 'Web form'
   WRONG 2.5 s Locate 'Web form'
   right 1.2 s Fill 'Text input' with 'Jane Doe'
   right 1.3 s Enter 'Jane Doe' in 'Text input'
   right 0.2 s Click on 'Submit'
   right 0.2 s Press 'Submit'
   right 2.4 s Double-click on 'File input'
   right 2.6 s Double click the 'File input' button
   right 0.2 s Right-click on 'Return to index'
   right 0.2 s Right click the 'Return to index' link
   WRONG 2.3 s Choose 'One' from 'Dropdown (select)'
   WRONG 2.2 s Select 'One' in the 'Dropdown (select)' dropdown
   right 2.3 s Check 'Checked radio'
   right 2.2 s Tick the 'Checked radio' box
   WRONG 2.6 s Enter 'hello world' in 'Readonly input'
   right 2.5 s Type 'hello world' into the 'Readonly input' field
   right 4.5 s Select the 'Default radio' option
   right 2.2 s Tick the 'Default radio' box
page: https://testpages.eviltester.com/styled/basic-html-form-test.html
   right 2.1 s Click the 'T

WARNING  [cdp_use.client] Received duplicate response for request 170 - ignoring


   WRONG 1.6 s Right-click on 'edit' under the 'Data Tables' section
   WRONG 1.5 s Move the mouse over 'edit' under the 'Data Tables' section
   WRONG 1.4 s Hover over 'edit' under the 'Data Tables' section
   WRONG 3.8 s Find the text 'First Name' under the 'Data Tables' section
   WRONG 3.4 s Locate 'First Name' under the 'Data Tables' section
   right 1.5 s Click on 'edit' under the 'Data Tables' section
   right 1.5 s Press 'edit' under the 'Data Tables' section
   right 0.2 s Double-click on 'Sauce Labs'
   right 0.3 s Double click the 'Sauce Labs' link
   WRONG 1.4 s Right-click on 'edit' under the 'Data Tables' section
   WRONG 1.4 s Right click the 'edit' link under the 'Data Tables' section
   WRONG 1.7 s Move the mouse over 'edit' under the 'Data Tables' section
   WRONG 1.4 s Hover over 'edit' under the 'Data Tables' section
   WRONG 3.1 s Locate 'Email' under the 'Data Tables' section
   WRONG 2.9 s Find the text 'Email' under the 'Data Tables' section
page: https://the-in

INFO     [BrowserSession] 📢 on_BrowserStopEvent - Calling reset() (force=True, keep_alive=True)
INFO     [BrowserSession] [SessionManager] Cleared all owned data (targets, sessions, mappings)
INFO     [BrowserSession] ✅ Browser session reset complete


   right 0.2 s Right click the 'Home' link


INFO     [BrowserSession] ✅ Browser session reset complete


# kpis of thunder x with laya browser

In [6]:
def percent(part, whole):
    if whole == 0:
        return "-"
    return str(round(100 * part / whole)) + " %"


total = len(results)
right = 0
answered = 0
right_when_answered = 0
right_action = 0
with_value = 0
right_value = 0
times = []

for r in results:
    times.append(r["seconds"])
    if r["right"]:
        right = right + 1
    if r["xpath"] is not None:
        answered = answered + 1
        if r["right"]:
            right_when_answered = right_when_answered + 1
    if r["action"] == r["expected_action"]:
        right_action = right_action + 1
    if r["expected_value"] is not None:
        with_value = with_value + 1
        if str(r["value"]).lower() == str(r["expected_value"]).lower():
            right_value = right_value + 1

times.sort()
print("rows           ", total)
print("accuracy       ", percent(right, total))
print("coverage       ", percent(answered, total))
print("precision      ", percent(right_when_answered, answered))
print("action accuracy", percent(right_action, total))
print("time average   ", round(sum(times) / total, 2), "s")
print("time median    ", round(times[total // 2], 2), "s")
print("time slowest   ", round(times[-1], 2), "s")

rows            178
accuracy        69 %
coverage        93 %
precision       75 %
action accuracy 78 %
time average    1.18 s
time median     0.59 s
time slowest    5.16 s


comparing to the other model

-2 % accuracy
-5 % coverage 
+7 % precision
+0.4 s time average
+0.04 s time median




# RUN 3 : LAYA BROWSER , WITH SYS ARCHITECTURE CHANGE TO FEED HIM DATA HE WAS TRAINED ON (in case matcher didnt find the answer)

In [6]:
browser = BrowserSession(headless=True, keep_alive=True)
await browser.start()
results = []

for url in pages:
    print("page:", url)
    session_id = http.post("/sessions", json={"url": url, "title": "kpi lab"}).json()["session_id"]
    await browser.navigate_to(url)
    await asyncio.sleep(2)
    page = await browser.get_current_page()

    position = 1
    for row in pages[url]:
        answer, seconds = ask_thunderx(session_id, position, row["instruction"])
        right = await is_right(page, answer.get("xpath"), row["expected_xpath"])

        result = {
            "instruction": row["instruction"],
            "category": row["category"],
            "expected_action": row["action"],
            "expected_value": row["value"],
            "expected_xpath": row["expected_xpath"],
            "action": answer.get("action"),
            "value": answer.get("value"),
            "xpath": answer.get("xpath"),
            "status": answer.get("status"),
            "route": answer.get("route"),
            "confidence": answer.get("confidence"),
            "seconds": seconds,
            "right": right,
        }
        results.append(result)

        if right:
            print("   right", round(seconds, 1), "s", row["instruction"])
        else:
            print("   WRONG", round(seconds, 1), "s", row["instruction"])
        position = position + 1

    http.delete("/sessions/" + session_id)

await browser.kill()

INFO     [BrowserSession] Setting viewport to 1920x1200 with device scale factor 1.0 whereas original device scale factor was 1.0


page: https://www.selenium.dev/selenium/web/web-form.html
   WRONG 2.8 s Find the text 'Web form'
   WRONG 3.2 s Locate 'Web form'
   right 1.9 s Fill 'Text input' with 'Jane Doe'
   right 1.6 s Enter 'Jane Doe' in 'Text input'
   right 0.2 s Click on 'Submit'
   right 0.1 s Press 'Submit'
   right 3.3 s Double-click on 'File input'
   right 2.8 s Double click the 'File input' button
   right 0.1 s Right-click on 'Return to index'
   right 0.1 s Right click the 'Return to index' link
   WRONG 2.8 s Choose 'One' from 'Dropdown (select)'
   WRONG 2.8 s Select 'One' in the 'Dropdown (select)' dropdown
   right 2.8 s Check 'Checked radio'
   right 3.3 s Tick the 'Checked radio' box
   WRONG 2.8 s Enter 'hello world' in 'Readonly input'
   right 2.9 s Type 'hello world' into the 'Readonly input' field
   right 2.7 s Select the 'Default radio' option
   right 2.7 s Tick the 'Default radio' box
page: https://testpages.eviltester.com/styled/basic-html-form-test.html
   right 3.8 s Click the 'T

WARNING  [cdp_use.client] Received duplicate response for request 186 - ignoring


   WRONG 3.8 s Hover over 'edit' under the 'Data Tables' section
   WRONG 4.5 s Locate 'Email' under the 'Data Tables' section
   WRONG 4.9 s Find the text 'Email' under the 'Data Tables' section
page: https://the-internet.herokuapp.com/hovers
   right 0.1 s Double click the 'Sauce Labs' link
   right 0.2 s Double-click on 'Sauce Labs'
   right 0.2 s Right click the 'Elemental Selenium' link
   right 0.1 s Right-click on 'Elemental Selenium'
   WRONG 1.6 s Locate 'Hover over the image for additional information'
   WRONG 1.7 s Find the text 'Hover over the image for additional information'
   WRONG 1.6 s Locate 'Hovers'
   WRONG 1.8 s Find the text 'Hovers'
page: https://demoqa.com/buttons
   right 0.2 s Click on 'Dynamic Properties'
   right 0.2 s Click the 'Dynamic Properties' link
   right 0.3 s Double-click on 'Links'
   right 0.2 s Double click the 'Links' link
   right 0.3 s Right-click on 'Upload and Download'
   right 0.2 s Right click the 'Upload and Download' link
   right 4.

WARNING  [cdp_use.client] Received duplicate response for request 242 - ignoring


   right 0.2 s Press 'Web Tables'
   right 0.2 s Click on 'Web Tables'
   right 7.4 s Double-click on 'Check Box'
   right 7.7 s Double click the 'Check Box' link
   right 0.2 s Right click the 'Links' link
   right 0.2 s Right-click on 'Links'
   right 3.8 s Hover over 'Upload and Download'
   right 3.8 s Move the mouse over 'Upload and Download'
   WRONG 7.8 s Locate 'Impressive'
   WRONG 7.7 s Find the text 'Impressive'
   right 0.3 s Check 'Impressive'
   right 7.9 s Tick the 'Impressive' box
   right 0.3 s Click on 'Dynamic Properties'
   right 0.2 s Click the 'Dynamic Properties' link
   WRONG 3.8 s Double-click on 'Radio Button'
   right 7.6 s Double click the 'Radio Button' link
   right 0.2 s Right click the 'Buttons' link
   right 0.3 s Right-click on 'Buttons'
   right 7.6 s Hover over 'Text Box'
   right 7.6 s Move the mouse over 'Text Box'
page: https://demoqa.com/text-box
   right 0.2 s Click on 'Buttons'
   right 0.3 s Press 'Buttons'
   right 0.3 s Double-click on 'Web 

INFO     [BrowserSession] 📢 on_BrowserStopEvent - Calling reset() (force=True, keep_alive=True)
INFO     [BrowserSession] [SessionManager] Cleared all owned data (targets, sessions, mappings)
INFO     [BrowserSession] ✅ Browser session reset complete


   right 0.2 s Right click the 'Home' link


INFO     [BrowserSession] ✅ Browser session reset complete


In [7]:
# KPIS 
def percent(part, whole):
    if whole == 0:
        return "-"
    return str(round(100 * part / whole)) + " %"


total = len(results)
right = 0
answered = 0
right_when_answered = 0
right_action = 0
with_value = 0
right_value = 0
times = []

for r in results:
    times.append(r["seconds"])
    if r["right"]:
        right = right + 1
    if r["xpath"] is not None:
        answered = answered + 1
        if r["right"]:
            right_when_answered = right_when_answered + 1
    if r["action"] == r["expected_action"]:
        right_action = right_action + 1
    if r["expected_value"] is not None:
        with_value = with_value + 1
        if str(r["value"]).lower() == str(r["expected_value"]).lower():
            right_value = right_value + 1

times.sort()
print("rows           ", total)
print("accuracy       ", percent(right, total))
print("coverage       ", percent(answered, total))
print("precision      ", percent(right_when_answered, answered))
print("action accuracy", percent(right_action, total))
print("time average   ", round(sum(times) / total, 2), "s")
print("time median    ", round(times[total // 2], 2), "s")
print("time slowest   ", round(times[-1], 2), "s")

rows            178
accuracy        70 %
coverage        94 %
precision       74 %
action accuracy 78 %
time average    2.08 s
time median     0.69 s
time slowest    7.87 s


| Metric          | Laya browser ; new architecture | old sys | Winner |
|-----------------|----------|----------|--------|
| Accuracy        | 70%      | 67%      | Laya browser     |
| Coverage        | 94%      | 98%      | sys     |
| Precision       | 74%      | 68%      | Laya browser     |
| Action accuracy | 78%      | 78%      | Tie    |
| Average time    | 2.08s    | 0.78s    | sys     |
| Median time     | 0.69s    | 0.54s    | sys     |
| Slowest         | 7.87s    | 3.51s    | sys     |